<a href="https://colab.research.google.com/github/Sharafatnoa/DailyPractice/blob/main/2026-10-22-statistics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Daily Practice — 2026-10-22 — Statistics: Is This Covariate Shift, or Just a Big Sample Size?

**Dataset:** [Wine Quality (UCI)](https://archive.ics.uci.edu/dataset/186/wine+quality) — 1,599 real
red wine samples and 4,898 real white wine samples, each with 11 lab-measured physicochemical
features (acidity, sugar, sulfur dioxide, pH, alcohol, etc.) plus a sensory `quality` score. Loaded
from the `jbrownlee/Datasets` public mirror of the classic UCI dataset.

## Problem statement

You're the QA / statistics reviewer for a wine-quality scoring service. The deployed model was
trained and validated **only on red wine** physicochemical measurements. The product team now wants
to route **white wine** samples through that same scoring endpoint without retraining, reasoning
that "wine is wine — the chemistry is basically the same." Before signing off on that change, you
need to determine, with statistical rigor rather than a glance at two `.describe()` tables, whether
the red and white wine feature distributions differ enough that feeding white wine into a
red-trained model counts as the kind of covariate shift that would invalidate the model's validated
performance claims.

**What you should produce**, for each of the 11 physicochemical features (everything except
`quality`):

1. A two-sample hypothesis test comparing its distribution in red wine vs. white wine.
2. A correction for running 11 tests at once, so a "significant" result isn't just noise from
   testing many features.
3. A nonparametric effect size for each feature — with n = 1,599 + 4,898 rows, even a practically
   meaningless difference will come back "statistically significant," so the p-value alone can't be
   the whole decision.
4. A decision rule that flags a feature as **materially shifted** only when it is *both*
   statistically significant (after correction) *and* practically large.
5. A bootstrapped 95% confidence interval on the effect size of the single most-shifted feature.
6. A short, specific recommendation — approve as-is, block, or approve-with-monitoring — naming the
   features that drove the decision.


## Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

RNG = np.random.default_rng(42)


In [ ]:
COLS = ["fixed_acidity", "volatile_acidity", "citric_acid", "residual_sugar", "chlorides",
        "free_sulfur_dioxide", "total_sulfur_dioxide", "density", "pH", "sulphates",
        "alcohol", "quality"]

RED_URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/winequality-red.csv"
WHITE_URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/winequality-white.csv"

red = pd.read_csv(RED_URL, header=None, names=COLS)
white = pd.read_csv(WHITE_URL, header=None, names=COLS)

FEATURES = [c for c in COLS if c != "quality"]

print("red wine:", red.shape, "  white wine:", white.shape)
red.head()


In [ ]:
# Quick look: side-by-side summary stats. Notice how close some of these look at a glance,
# and how far apart others are -- that's exactly why we need a test, not a stare.
summary = pd.concat(
    {"red": red[FEATURES].describe().T[["mean", "std"]],
     "white": white[FEATURES].describe().T[["mean", "std"]]},
    axis=1,
)
summary


## Your turn: implement the five functions below

In [ ]:
def mannwhitney_table(red_df, white_df, features):
    """
    TODO: For each feature name in `features`, run a two-sided Mann-Whitney U
    test (scipy.stats.mannwhitneyu) comparing red_df[feature] vs white_df[feature].

    Use the Mann-Whitney U test rather than a t-test: several of these
    features (e.g. residual_sugar, chlorides) are heavily skewed, not
    normally distributed.

    Return a DataFrame with one row per feature and columns:
    "feature", "u_stat", "p_value" (in the same order as `features`).
    """
    raise NotImplementedError


def fdr_correct(p_values, alpha=0.05):
    """
    TODO: Implement the Benjamini-Hochberg false discovery rate procedure.

    Given an array of m p-values from independent tests:
      1. Sort them ascending: p_(1) <= p_(2) <= ... <= p_(m).
      2. Find the largest rank k such that p_(k) <= (k / m) * alpha.
      3. Reject the null hypothesis for every test with rank <= k.
      4. Also compute BH-adjusted p-values: adjusted_(k) = p_(k) * m / k, then
         make them monotone non-increasing from the largest rank down, and
         clip to [0, 1].

    Return (adjusted_p_values, reject_flags) as two numpy arrays in the
    SAME order as the input `p_values` (not sorted order).

    Why this matters: testing 11 features at alpha=0.05 each gives roughly a
    1 - 0.95**11 ≈ 43% chance of at least one false "significant" result
    even if none of the features actually differ. FDR correction controls
    for that.
    """
    raise NotImplementedError


def cliffs_delta(x, y):
    """
    TODO: Compute Cliff's delta, a nonparametric effect size for how much
    x's values tend to exceed y's values. It ranges from -1 (every x < every
    y) to +1 (every x > every y), with 0 meaning no tendency either way.

    Hint: Cliff's delta has a direct relationship to the Mann-Whitney U
    statistic computed on (x, y): delta = (2 * U) / (n1 * n2) - 1, where U
    is the statistic scipy.stats.mannwhitneyu(x, y) returns for the first
    sample passed in.
    """
    raise NotImplementedError


def bootstrap_effect_size_ci(x, y, effect_fn=cliffs_delta, n_boot=2000, alpha=0.05, rng=RNG):
    """
    TODO: Bootstrap a (1 - alpha) confidence interval for effect_fn(x, y).

    Resample x (with replacement, same size as x) and resample y (with
    replacement, same size as y) independently, n_boot times, recompute
    effect_fn on each pair of resamples, and return
    (point_estimate, lower, upper) using the percentile method.
    """
    raise NotImplementedError


def flag_shifted_features(results_df, alpha=0.05, delta_threshold=0.2):
    """
    TODO: Given a results DataFrame with columns "p_adj" and "cliffs_delta",
    add a boolean "flagged" column that is True only when BOTH:
      - the feature is statistically significant after correction
        (p_adj < alpha), AND
      - the effect size is practically large (abs(cliffs_delta) >= delta_threshold).

    Return the DataFrame with the new column added.
    """
    raise NotImplementedError


In [ ]:
# TODO: put it together.
# 1. Call mannwhitney_table(red, white, FEATURES).
# 2. Run fdr_correct on the resulting p_values and store the adjusted
#    p-values and reject flags as new columns ("p_adj", "reject_bh").
# 3. Add a "cliffs_delta" column: cliffs_delta(red[feature], white[feature])
#    for each feature.
# 4. Call flag_shifted_features(...) and sort by |cliffs_delta| descending.
# 5. Print the full results table.
# 6. For the top-flagged feature, call bootstrap_effect_size_ci and print the
#    point estimate and 95% CI.
# 7. Print a short recommendation: approve / block / approve-with-monitoring,
#    naming the features that drove the decision.


---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

### Implementation

```python
def mannwhitney_table(red_df, white_df, features):
    rows = []
    for f in features:
        x, y = red_df[f].values, white_df[f].values
        u_stat, p_value = stats.mannwhitneyu(x, y, alternative="two-sided")
        rows.append({"feature": f, "u_stat": u_stat, "p_value": p_value})
    return pd.DataFrame(rows)


def fdr_correct(p_values, alpha=0.05):
    p = np.asarray(p_values)
    m = len(p)
    order = np.argsort(p)
    ranked = p[order]

    thresholds = (np.arange(1, m + 1) / m) * alpha
    passed = ranked <= thresholds
    cutoff = ranked[np.max(np.where(passed)[0])] if passed.any() else 0.0
    reject_sorted = ranked <= cutoff

    adj_sorted = ranked * m / np.arange(1, m + 1)
    adj_sorted = np.minimum.accumulate(adj_sorted[::-1])[::-1]  # monotone
    adj_sorted = np.clip(adj_sorted, 0, 1)

    reject = np.empty(m, dtype=bool)
    adjusted = np.empty(m)
    reject[order] = reject_sorted
    adjusted[order] = adj_sorted
    return adjusted, reject


def cliffs_delta(x, y):
    x, y = np.asarray(x), np.asarray(y)
    n1, n2 = len(x), len(y)
    u_stat, _ = stats.mannwhitneyu(x, y, alternative="two-sided")
    return (2 * u_stat) / (n1 * n2) - 1


def bootstrap_effect_size_ci(x, y, effect_fn=cliffs_delta, n_boot=2000, alpha=0.05, rng=RNG):
    x, y = np.asarray(x), np.asarray(y)
    n1, n2 = len(x), len(y)
    boots = np.empty(n_boot)
    for i in range(n_boot):
        xb = x[rng.integers(0, n1, n1)]
        yb = y[rng.integers(0, n2, n2)]
        boots[i] = effect_fn(xb, yb)
    point = effect_fn(x, y)
    lower, upper = np.percentile(boots, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return point, lower, upper


def flag_shifted_features(results_df, alpha=0.05, delta_threshold=0.2):
    sig = results_df["p_adj"] < alpha
    big = results_df["cliffs_delta"].abs() >= delta_threshold
    out = results_df.copy()
    out["flagged"] = sig & big
    return out
```

### Putting it together

```python
mw = mannwhitney_table(red, white, FEATURES)
mw["p_adj"], mw["reject_bh"] = fdr_correct(mw["p_value"].values)
mw["cliffs_delta"] = [cliffs_delta(red[f], white[f]) for f in FEATURES]
mw = flag_shifted_features(mw)
mw = mw.sort_values("cliffs_delta", key=lambda s: s.abs(), ascending=False)
print(mw.to_string(index=False))

top_feature = mw.iloc[0]["feature"]
point, lo, hi = bootstrap_effect_size_ci(red[top_feature], white[top_feature])
print(f"\ntop shifted feature: {top_feature}")
print(f"Cliff's delta = {point:.3f}  95% CI [{lo:.3f}, {hi:.3f}]")
print(f"flagged features: {mw['flagged'].sum()} / {len(mw)}")
```

This typically prints a table like:

```
             feature     u_stat        p_value          p_adj  reject_bh  cliffs_delta  flagged
total_sulfur_dioxide   366639.5   0.000000e+00   0.000000e+00       True     -0.906         True
           chlorides  7407015.5   0.000000e+00   0.000000e+00       True      0.891         True
    volatile_acidity  7059623.5   0.000000e+00   0.000000e+00       True      0.803         True
 free_sulfur_dioxide  1186396.5   0.000000e+00   0.000000e+00       True     -0.697         True
           sulphates  6509961.0   0.000000e+00   0.000000e+00       True      0.662         True
       fixed_acidity  6138507.0  1.439e-255      2.638e-255        True      0.568         True
             density  6059284.5  1.453e-237      2.283e-237        True      0.547         True
                  pH  5681839.5  5.472e-162      7.524e-162        True      0.451         True
      residual_sugar  2569687.0   5.634e-95       6.886e-95        True     -0.344         True
         citric_acid  3070088.5   1.313e-38       1.444e-38        True     -0.216         True
             alcohol  3829043.5   1.818e-01       1.818e-01       False     -0.022        False

top shifted feature: total_sulfur_dioxide
Cliff's delta = -0.906  95% CI [-0.918, -0.894]
flagged features: 10 / 11
```

**Recommendation:** block the change. 10 of the 11 physicochemical features show a
statistically significant (after FDR correction) *and* practically huge shift between red and
white wine (several `|Cliff's delta|` values above 0.5 — white wine is, chemically, a very
different population, which is unsurprising given how differently it's made: longer skin contact
for red, more sulfur dioxide as a preservative for white, more residual sugar in many whites, and
so on). A model trained and validated only on red wine chemistry has no evidence behind it for
inputs this different; routing white wine through it would be scoring out-of-distribution inputs
and reporting the red-wine validation numbers as if they still applied. The one exception,
`alcohol`, is genuinely *not* shifted (p_adj ≈ 0.18, delta ≈ -0.02) — which is itself a useful,
specific finding: if the business case only needs an alcohol-driven sub-model, that slice might be
safe to extend; the rest of the feature space is not.

### Why both the corrected p-value *and* the effect size matter

With n = 1,599 + 4,898, almost any real difference, however small, will produce a tiny p-value —
that's what large samples are for. If this notebook stopped at step 2 (p-value only), it would
flag `alcohol` as "significant" too under an uncorrected single test, and it would treat a Cliff's
delta of 0.02 the same as one of 0.91. That's the statistical equivalent of treating a 0.1%
flaky-test rate the same as a consistently broken build: both might clear some p < 0.05 bar, but
only one of them is an engineering problem. Combining statistical significance (is this probably
not noise?) with effect size (is it big enough to care about?) is what turns a wall of asterisks
into an actual go/no-go decision — the same discipline a monitoring dashboard needs so it pages
someone for a real regression, not a rounding error.

</details>